# មេរៀន ១៧ - បង្កើតភ្នាក់ងារតំបន់ AI ជាមួយ Foundry Local និង Qwen

ក្នុងកំណត់ហេតុនេះ អ្នកបានបង្កើត **ជំនួយការបច្ចេកវិជ្ជាផ្ទាល់មួយ** ដែលដំណើរការទាំងមូលលើកុំព្យូទ័ររបស់អ្នក។ មិនមានការបក្យាខ្យល់ពីពពកទេនៅពេលណាមួយឡើយ។ ជំនួយការអាច:

1. **ហៅឧបករណ៍** តាមរយៈការហៅមុខងារ Qwen ក្រោម Foundry Local។
2. **បញ្ជីនិងអានឯកសារ** នៅក្នុងថតគម្រោងដែលមានការកំចាត់សុវត្ថិភាព។
3. **វិភាគកូដ** ជាមួយវិមាត្រងាយៗ។
4. **ស្វែងរកឯកសារយោង** ជាមួយ RAG តំបន់មូលដ្ឋាន (Chroma)។
5. **ប្រើម៉ាស៊ីនបម្រើ MCP តំបន់មូលដ្ឋាន** (រំកិលយ៉ាងបរិញ្ញាណប្រសិនបើគ្មានដែលបានកំណត់)។

កូដភ្នាក់ងារមើលទៅស្រដៀងគ្នាប្រហែលមួយនឹងមេរៀននៅពពក — មានតែចំណុចផ្លូវភ្នែកម៉ាស៊ីនអតិថិជនផ្លាស់ទីពីពពកទៅ `localhost` เท่านั้น។


## ការតំឡើង

មុនពេលដំណើរការសៀវភៅចតុកោណនេះ៖

1. **ដំឡើង Microsoft Foundry Local** (មើល [ឯកសារ](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) សម្រាប់ប្រព័ន្ធប្រតិបត្តិការរបស់អ្នក)។
2. **ទាញយកនិងចាប់ផ្តើមម៉ូដែល Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. ដំឡើងកញ្ចប់ Python ខាងក្រោម។

អ្វីៗទាំងអស់ដំណើរការរួចជាស្រុក។ កុំព្យូទ័រមួយដែលមាន RAM ~8 GB គឺជាកម្រិតអប្បបរមាដែលអាចធ្វើបាន។


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. ចូលទៅកាន់ Foundry Local

`FoundryLocalManager` ទាញយកម៉ូដែលបើការទាមទារមាន ត្រូវចាប់ផ្តើមសេវាកម្មក្នុងមូលដ្ឋាន និងផ្តល់ឱ្យយើងនូវ **ចំណុចទំនាក់ទំនងដែលផ្គូផ្គងជាមួយ OpenAI**។ បន្ទាប់មក យើងបញ្ជូន SDK OpenAI ស្តង់ដារទៅកាន់វា។ កូនសោ API គឺជាកន្លែងកាន់កាប់ក្នុងមូលដ្ឋាន — គ្មានលិខិតអាកាសម៉េគ័រណាអ្វីមួយត្រូវបានចូលរួមជាមួយ។


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. ឧបករណ៍ក្នុងតំបន់ (ប្រតិបត្តិការឯកសារដោយមានដែនកំណត់)

យើងបង្កើតគំរូគម្រោងតូចមួយលើឌីសបន្ទាប់មកកំណត់ឧបករណ៍ដែលមានដែនកំណត់ទៅឫសគម្រោងនោះ។ ការត្រួតពិនិត្យsandbox មានសារៈសំខាន់ថែមទាំងនៅក្នុងតំបន់ផ្ទាល់៖ ឧបករណ៍ដែលអានផ្លូវណាមួយដោយចៃដន្យនោះដំណើរការជាមួយការអនុញ្ញាតអ្នកប្រើរបស់អ្នក និងអាចប៉ះពាល់អ្វីក៏បានដែលអ្នកអាចធ្វើបាន។ 


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. ការរកស៊ីត្រូវតាមតំបន់ជាមួយ Chroma

យើងបញ្ចូលសំណុំចំណុចឯកសារតូចៗមួយចំនួនទៅក្នុងការប្រមូល Chroma ជាតំបន់។ Chroma បានរត់ក្នុងដំណើរការនិងរក្សាទុកវ៉ិចទ័រនៅលើឌីស — គ្មានម៉ាស៊ីនបម្រើ មិនមានពពកទេ។ ឧបករណ៍ `search_docs` នាំយកចំណុចដែលសមរម្យបំផុតសម្រាប់សំណួរ។


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. វដ្តការហៅឧបករណ៍

ឥឡូវនេះ យើងបានចុះបញ្ជីឧបករណ៍ជាមួយម៉ូដែលដោយប្រើស្កីម៉ាទិកឧបករណ៍ OpenAI ហើយដំណើរការវដ្តហៅឧបករណ៍ស្តង់ដារ — ម៉ូដែលស្នើសុំឧបករណ៍ យើងអនុវត្តវានៅក្នុងផ្ទះ ស្វែងរកលទ្ធផលវិញ ហើយធ្វើម្តងទៀតរហូតម៉ូដែលបញ្ចេញចំលើយចុងក្រោយ។ ការហៅមុខងារដែលអាចនិយោជប្រកបដោយទំនុកចិត្តរបស់ Qwen គឺជាមូលហេតុដែលធ្វើឲ្យវាសម្ដេចក្នុងឧបករណ៍។


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP ក្នុងសហគ្រិន (ជាជម្រើស)

MCP គឺជាប្រព័ន្ធផ្ទេរទិន្នន័យ មិនមែនជាសេវាកម្មពពកទេ — ម៉ាស៊ីនម៉ាស៊ីន MCP អាចដំណើរការជាវិធានការរបស់អ្នកនៅលើ `stdio`។ បន្ទាត់ខាងក្រោមបង្ហាញពីរបៀបដែលអ្នកអាចភ្ជាប់ទៅម៉ាស៊ីន MCP ក្នុងសហគ្រិន ប្រសិនបើអ្នកមានការកំណត់វា (ជាឧទាហរណ៏ ម៉ាស៊ីនម៉ាស៊ីនប្រព័ន្ធប្រេីប្រាស់ឯកសារ)។ វាចោលដោយសុវត្ថិភាពនៅពេល `LOCAL_MCP_COMMAND` មិនត្រូវបានកំណត់ ដូច្នេះសៀវភៅកំណត់ត្រានៅតែដំណើរការពេញលេញដោយគ្មានវា។

សម្គាល់សន្តិសុខ៖ ម៉ាស៊ីន MCP ក្នុងសហគ្រិនដំណើរការជាមួយសិទិ្ធអ្នកប្រើរបស់អ្នក។ កំណត់វាទៅកាន់ថតគម្រោង និងផ្ទៀងផ្ទាត់លទ្ធផលរបស់វាមុនពេលអនុវត្តតាមវា។


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## សារសង្ខេប

អ្នកបានបង្កើតជំនួយការជំនាញវិស្វកម្មដែលដំណើរការជាសរុបលើម៉ាស៊ីនរបស់អ្នក:

- **Foundry Local** បានផ្តល់ម៉ូដែល **Qwen** នៅពីក្រោយចុងបញ្ចប់ដែលឆបគ្នាជាមួយ OpenAI — ដូច្នេះកូដភ្នាក់ងារចូលគ្នានឹងមេរៀនក្នុងពពក។
- **ឧបករណ៍ Sandboxed** បានផ្តល់ឱ្យភ្នាក់ងារមានការចូលដំណើរការឯកសារ និងវិភាគកូដដោយគ្មានការចេញពីថតគម្រោង។
- **Chroma** បានផ្តល់ **RAG ផ្ទាល់ខ្លួន** លើឯកសារយោង។
- **Local MCP** បានបង្ហាញពីរបៀបប្រើប្រាស់សេវាកម្ម MCP ជាផ្ទាល់ក្រៅអនឡាញ។

មិនបានប្រើការប៉ាន់ស្មានពពកនៅពេលណាមួយទេ។

### បញ្ហា

ពង្រីកភ្នាក់ងារផ្ទាល់ខ្លួនដើម្បី:

1. **ធ្វើការជាមួយម៉ាស៊ីនមេ MCP ច្រើនមួយ** — ជួបភ្ជាប់ម៉ាស៊ីនមេប្រព័ន្ធឯកសារ និងម៉ាស៊ីនមេ git ហើយអនុញ្ញាតឱ្យភ្នាក់ងារជ្រើសរើសរវាងពួកវា។
2. **ប្រើសមត្ថភាពចងចាំក្នុងជ្រុលខ្លួន** — រក្សានូវប្រវត្តិសន្ទនាសង្កត់ខ្លីទៅឌីស្ក์ ដូច្នេះជំនួយករត្រៀមចងចាំការប្រាស្រ័យទាក់ទងពីមុនៗកាលពីបើកសៀវភៅកំណត់ត្រាឡើងវិញ។
3. **គាំទ្រការរៀបចំការងារភ្នាក់ងារច្រើនជាស្រុកខ្លួន** — បន្ថែមភ្នាក់ងារទីពីរជាស្រុកខ្លួន (ឧ. អ្នកពិនិត្យ) ហើយឱ្យមនុស្សទាំងពីរសហការលើបេសកកម្មមួយ។

នៅមេរៀនបន្ទាប់ អ្នកនឹងរៀនពីរបៀបធានាសុវត្ថិភាពភ្នាក់ងារត្រូវបានដាក់ចេញផ្សាយ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
